In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
project_dir = Path.home() / "Desktop" / "8730 _ Group Assignment"

raw_dir = project_dir / "market_data"

processed_dir = project_dir / "processed"

processed_dir.mkdir(exist_ok=True)

print("Raw data folder:", raw_dir)
print("Folder exists:", raw_dir.exists())

for file in raw_dir.glob("*.csv"):
    print(file.name)

Raw data folder: /Users/yuanquan0901/Desktop/8730 _ Group Assignment/market_data
Folder exists: True
SmartCentres_daily.csv
REIT_Benchmark_daily.csv
Choice_Properties_daily.csv
RioCan_daily.csv


In [7]:
smartcentres = pd.read_csv(raw_dir / "SmartCentres_daily.csv")

riocan = pd.read_csv(raw_dir / "RioCan_daily.csv")

choice = pd.read_csv(raw_dir / "Choice_Properties_daily.csv")

benchmark = pd.read_csv(raw_dir / "REIT_Benchmark_daily.csv")

datasets = {
    "SmartCentres": smartcentres,
    "RioCan": riocan,
    "Choice Properties": choice,
    "REIT Benchmark": benchmark
}

for name, df in datasets.items():
    print(name, df.shape)

SmartCentres (941, 9)
RioCan (941, 9)
Choice Properties (941, 9)
REIT Benchmark (941, 10)


In [9]:
for name, df in datasets.items():

    print("=" * 60)
    print("Dataset:", name)
    print("=" * 60)

    print("Number of rows:", len(df))
    print("Number of columns:", len(df.columns))

    print("\nDate Range:")
    print("Start:", df["Date"].min())
    print("End:", df["Date"].max())

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nDuplicate Dates:")
    print(df["Date"].duplicated().sum())

    print("\nData Types:")
    print(df.dtypes)

    print("\n")

Dataset: SmartCentres
Number of rows: 941
Number of columns: 9

Date Range:
Start: 2023-01-03 00:00:00-05:00
End: 2026-09-30 00:00:00-04:00

Missing Values:
Date            0
Open            0
High            0
Low             0
Close           0
Adj Close       0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64

Duplicate Rows:
0

Duplicate Dates:
0

Data Types:
Date             object
Open            float64
High            float64
Low             float64
Close           float64
Adj Close       float64
Volume            int64
Dividends       float64
Stock Splits    float64
dtype: object


Dataset: RioCan
Number of rows: 941
Number of columns: 9

Date Range:
Start: 2023-01-03 00:00:00-05:00
End: 2026-09-30 00:00:00-04:00

Missing Values:
Date            0
Open            0
High            0
Low             0
Close           0
Adj Close       0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64

Duplicate Rows:
0

Duplicate Dates:
0

Data Types:
Date  

In [11]:
tickers = {
    "SmartCentres": "SRU-UN.TO",
    "RioCan": "REI-UN.TO",
    "Choice Properties": "CHP-UN.TO",
    "REIT Benchmark": "XRE.TO"
}

cleaned_datasets = []

for name, df in datasets.items():

    df = df.copy()

    # Standardize date format
    df["Date"] = pd.to_datetime(df["Date"].str[:10])

    # Standardize column names
    df.columns = df.columns.str.lower().str.replace(" ", "_")

    # Add company name and ticker
    df["company"] = name
    df["ticker"] = tickers[name]
    df["currency"] = "CAD"

    # Standardize columns
    if "capital_gains" not in df.columns:
        df["capital_gains"] = np.nan

    cleaned_datasets.append(df)

    print(name, "- Standardization completed")

SmartCentres - Standardization completed
RioCan - Standardization completed
Choice Properties - Standardization completed
REIT Benchmark - Standardization completed


In [13]:
for df in cleaned_datasets:

    print("=" * 50)
    print("Company:", df["company"].iloc[0])

    print("Invalid Dates:",
          df["date"].isna().sum())

    print("Duplicate Dates:",
          df["date"].duplicated().sum())

    print("Invalid Prices:",
          (df[["open", "high", "low",
               "close", "adj_close"]] <= 0).any(axis=1).sum())

    print("Negative Volume:",
          (df["volume"] < 0).sum())

    print("Negative Dividends:",
          (df["dividends"] < 0).sum())

    print("Invalid High/Low:",
          (df["high"] < df["low"]).sum())

    print("Invalid OHLC:",
          ((df["high"] < df[["open", "close"]].max(axis=1)) |
           (df["low"] > df[["open", "close"]].min(axis=1))).sum())

Company: SmartCentres
Invalid Dates: 0
Duplicate Dates: 0
Invalid Prices: 0
Negative Volume: 0
Negative Dividends: 0
Invalid High/Low: 0
Invalid OHLC: 0
Company: RioCan
Invalid Dates: 0
Duplicate Dates: 0
Invalid Prices: 0
Negative Volume: 0
Negative Dividends: 0
Invalid High/Low: 0
Invalid OHLC: 0
Company: Choice Properties
Invalid Dates: 0
Duplicate Dates: 0
Invalid Prices: 0
Negative Volume: 0
Negative Dividends: 0
Invalid High/Low: 0
Invalid OHLC: 0
Company: REIT Benchmark
Invalid Dates: 0
Duplicate Dates: 0
Invalid Prices: 0
Negative Volume: 0
Negative Dividends: 0
Invalid High/Low: 0
Invalid OHLC: 0


In [15]:
market_data = pd.concat(
    cleaned_datasets,
    ignore_index=True
)

market_data = market_data.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

print("Final Shape:", market_data.shape)

print("Duplicate Primary Keys:",
      market_data.duplicated(["ticker", "date"]).sum())

market_data.head(10)

Final Shape: (3764, 13)
Duplicate Primary Keys: 0


,date,open,high,low,close,adj_close,volume,dividends,stock_splits,company,ticker,currency,capital_gains
0,2023-01-03,14.79,14.970,14.750,14.91,12.140608,537900,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
1,2023-01-04,14.99,15.090,14.910,15.00,12.213893,271300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
2,2023-01-05,14.94,14.940,14.765,14.85,12.091756,472200,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
3,2023-01-06,14.90,15.040,14.830,14.95,12.173180,406300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
4,2023-01-09,14.87,15.100,14.790,14.94,12.165037,290300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
5,2023-01-10,14.93,15.050,14.930,15.05,12.254609,294300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
6,2023-01-11,15.08,15.440,15.080,15.40,12.539598,619100,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
7,2023-01-12,15.50,15.585,15.350,15.43,12.564025,588800,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
8,2023-01-13,15.40,15.570,15.320,15.54,12.653596,352800,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
9,2023-01-16,15.50,15.580,15.400,15.50,12.621026,241600,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN


In [17]:
# Final data validation

print("Final Dataset Shape:", market_data.shape)

print("\nRecords per Company:")
print(market_data.groupby("company").size())

print("\nDate Range:")
print(market_data["date"].min())
print(market_data["date"].max())

print("\nMissing Values:")
print(market_data.isnull().sum())

print("\nDuplicate Primary Keys:")
print(market_data.duplicated(["ticker", "date"]).sum())

print("\nTrading Date Consistency:")
date_sets = [
    set(df["date"])
    for df in cleaned_datasets
]

print("Identical Trading Dates:",
      all(d == date_sets[0] for d in date_sets))

print("\nData Types:")
print(market_data.dtypes)

Final Dataset Shape: (3764, 13)

Records per Company:
company
Choice Properties    941
REIT Benchmark       941
RioCan               941
SmartCentres         941
dtype: int64

Date Range:
2023-01-03 00:00:00
2026-09-30 00:00:00

Missing Values:
date                0
open                0
high                0
low                 0
close               0
adj_close           0
volume              0
dividends           0
stock_splits        0
company             0
ticker              0
currency            0
capital_gains    2823
dtype: int64

Duplicate Primary Keys:
0

Trading Date Consistency:
Identical Trading Dates: True

Data Types:
date             datetime64[ns]
open                    float64
high                    float64
low                     float64
close                   float64
adj_close               float64
volume                    int64
dividends               float64
stock_splits            float64
company                  object
ticker                   object
currenc

In [19]:
# Export cleaned market data

output_file = processed_dir / "market_data_cleaned.csv"

market_data.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d"
)

print("Export Successful:", output_file.exists())
print("Saved to:", output_file)
print("Total Records:", len(market_data))

Export Successful: True
Saved to: /Users/yuanquan0901/Desktop/8730 _ Group Assignment/processed/market_data_cleaned.csv
Total Records: 3764


In [21]:
saved_data = pd.read_csv(output_file)

print("Saved Data Shape:", saved_data.shape)

print("Export Validation:",
      saved_data.shape == market_data.shape)

saved_data.head()

Saved Data Shape: (3764, 13)
Export Validation: True


,date,open,high,low,close,adj_close,volume,dividends,stock_splits,company,ticker,currency,capital_gains
0,2023-01-03,14.79,14.97,14.750,14.91,12.140608,537900,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
1,2023-01-04,14.99,15.09,14.910,15.00,12.213893,271300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
2,2023-01-05,14.94,14.94,14.765,14.85,12.091756,472200,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
3,2023-01-06,14.90,15.04,14.830,14.95,12.173180,406300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
4,2023-01-09,14.87,15.10,14.790,14.94,12.165037,290300,0.0,0.0,Choice Properties,CHP-UN.TO,CAD,NaN
